In [1]:
# import necessary libraries

import numpy as np
import pickle
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

In [2]:
# load citeseer data

data = np.load(
    "citeseer_prepared.npz",
    allow_pickle=True
)

X = data["X"]
y = data["y"]

train_indices = data["train_indices"]
test_indices = data["test_indices"]

node_ids = data["node_ids"]

In [3]:
# load neighborhood dictionary
with open("citeseer_neighbors.pkl", "rb") as f:
    neighbors = pickle.load(f)

# create node-to-index mapping
node_to_index = {
    node_id: index
    for index, node_id in enumerate(node_ids)
}

In [4]:
# get center node and neighbors
def get_node_data(node_id):
    center_index = node_to_index[node_id]

    center_features = X[center_index]

    neighbor_ids = list(neighbors[node_id])

    neighbor_features = np.array([
        X[node_to_index[neighbor_id]]
        for neighbor_id in neighbor_ids
    ])

    label = y[center_index]

    return center_features, neighbor_features, label

In [5]:
# test on one node

center_features, neighbor_features, label = get_node_data(
    node_ids[0]
)

print("Center shape:", center_features.shape)
print("Neighbors shape:", neighbor_features.shape)
print("Label:", label)

Center shape: (3703,)
Neighbors shape: (12, 3703)
Label: 1


In [6]:
# create citeseer dataset
class CiteseerDataset(Dataset):
    def __init__(self, indices):
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        node_index = self.indices[idx]
        node_id = node_ids[node_index]

        center_features, neighbor_features, label = get_node_data(
            node_id
        )

        return center_features, neighbor_features, label

In [7]:
# create train-test dataset
train_dataset = CiteseerDataset(train_indices)
test_dataset = CiteseerDataset(test_indices)

print("Training samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

Training samples: 2649
Test samples: 663


In [8]:
# create collate function
def citeseer_collate(batch):
    centers = []
    neighbor_features = []
    labels = []

    for center, neighbor_set, label in batch:
        centers.append(
            torch.tensor(
                center,
                dtype=torch.float32
            )
        )

        neighbor_features.append(
            torch.tensor(
                neighbor_set,
                dtype=torch.float32
            )
        )

        labels.append(label)

    centers = torch.stack(centers)

    labels = torch.tensor(
        labels,
        dtype=torch.long
    )

    return centers, neighbor_features, labels

In [9]:
# create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=citeseer_collate
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=citeseer_collate
)

In [10]:
# test it on one batch
centers, neighbor_features, labels = next(
    iter(train_loader)
)

print("Centers shape:", centers.shape)
print("Number of neighbor sets:", len(neighbor_features))
print(
    "First neighbor set shape:",
    neighbor_features[0].shape
)
print("Labels shape:", labels.shape)

Centers shape: torch.Size([16, 3703])
Number of neighbor sets: 16
First neighbor set shape: torch.Size([6, 3703])
Labels shape: torch.Size([16])


In [11]:
# create Citeseer PointNet++ model
class CiteseerPointNetPlusPlus(nn.Module):
    def __init__(
        self,
        input_dim=3703,
        hidden_dim=64,
        num_classes=6
    ):
        super().__init__()

        # First local feature abstraction
        self.input_projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU()
        )

        self.local_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )

        # Second feature abstraction
        self.second_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )

        # Center node representation
        self.center_embedding = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU()
        )

        # Final classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim + hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, center, neighbor_features):

        # First abstraction
        neighbor_features = self.input_projection(
            neighbor_features
        )

        neighbor_features = self.local_mlp(
            neighbor_features
        )

        # Symmetric aggregation
        pooled = neighbor_features.max(dim=0).values

        # Second abstraction
        pooled = self.second_mlp(
            pooled
        )

        # Center representation
        center_features = self.center_embedding(
            center
        )

        # Combine center and neighborhood
        combined = torch.cat(
            [
                center_features,
                pooled
            ],
            dim=0
        )

        # Classification
        output = self.classifier(combined)

        return output

In [12]:
# create the model and inspect it
model = CiteseerPointNetPlusPlus()

print(model)

CiteseerPointNetPlusPlus(
  (input_projection): Sequential(
    (0): Linear(in_features=3703, out_features=64, bias=True)
    (1): ReLU()
  )
  (local_mlp): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
  )
  (second_mlp): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
  )
  (center_embedding): Sequential(
    (0): Linear(in_features=3703, out_features=64, bias=True)
    (1): ReLU()
  )
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=6, bias=True)
  )
)


In [13]:
# inspect its parameters
total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Trainable parameters:", total_params)

Trainable parameters: 499398


In [14]:
# test one forward pass
center, neighbor_set, label = train_dataset[0]

center = torch.tensor(
    center,
    dtype=torch.float32
)

neighbor_set = torch.tensor(
    neighbor_set,
    dtype=torch.float32
)

output = model(
    center,
    neighbor_set
)

print("Center shape:", center.shape)
print("Neighbors shape:", neighbor_set.shape)
print("Output shape:", output.shape)
print("True label:", label)

Center shape: torch.Size([3703])
Neighbors shape: torch.Size([1, 3703])
Output shape: torch.Size([6])
True label: 0


In [15]:
# create loss function and optimizer
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [16]:
# training loop for 30 epochs
num_epochs = 30

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0

    for centers, neighbor_features, labels in train_loader:

        optimizer.zero_grad()

        batch_loss = 0.0

        for i in range(len(centers)):

            output = model(
                centers[i],
                neighbor_features[i]
            )

            loss = criterion(
                output.unsqueeze(0),
                labels[i].unsqueeze(0)
            )

            batch_loss += loss

        batch_loss = batch_loss / len(centers)

        batch_loss.backward()
        optimizer.step()

        running_loss += batch_loss.item()

    average_loss = running_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1}/{num_epochs}, "
        f"Loss: {average_loss:.4f}"
    )

Epoch 1/30, Loss: 1.2833
Epoch 2/30, Loss: 0.4705
Epoch 3/30, Loss: 0.1697
Epoch 4/30, Loss: 0.0553
Epoch 5/30, Loss: 0.0209
Epoch 6/30, Loss: 0.0144
Epoch 7/30, Loss: 0.0117
Epoch 8/30, Loss: 0.0211
Epoch 9/30, Loss: 0.0103
Epoch 10/30, Loss: 0.0100
Epoch 11/30, Loss: 0.0017
Epoch 12/30, Loss: 0.0027
Epoch 13/30, Loss: 0.0033
Epoch 14/30, Loss: 0.0006
Epoch 15/30, Loss: 0.0005
Epoch 16/30, Loss: 0.0004
Epoch 17/30, Loss: 0.0003
Epoch 18/30, Loss: 0.0003
Epoch 19/30, Loss: 0.0002
Epoch 20/30, Loss: 0.0002
Epoch 21/30, Loss: 0.0002
Epoch 22/30, Loss: 0.0001
Epoch 23/30, Loss: 0.0001
Epoch 24/30, Loss: 0.0001
Epoch 25/30, Loss: 0.0001
Epoch 26/30, Loss: 0.0001
Epoch 27/30, Loss: 0.0001
Epoch 28/30, Loss: 0.0001
Epoch 29/30, Loss: 0.0001
Epoch 30/30, Loss: 0.0001


In [17]:
# test accuracy
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for centers, neighbor_features, labels in test_loader:

        for i in range(len(centers)):

            output = model(
                centers[i],
                neighbor_features[i]
            )

            prediction = torch.argmax(
                output
            ).item()

            all_predictions.append(prediction)
            all_labels.append(labels[i].item())

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

print(f"Test accuracy: {accuracy:.4f}")
print(f"Test accuracy: {accuracy * 100:.2f}%")

Test accuracy: 0.7526
Test accuracy: 75.26%


In [18]:
# confusion matrix

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("Confusion matrix:")
print(cm)

Confusion matrix:
[[ 21   6   6   4   1  12]
 [  5 104   1   3   1   5]
 [  3   2 111   2  15   7]
 [  2  10   2  82   4   2]
 [  2   7   7   9  93  16]
 [  7   3   5   4  11  88]]


In [19]:
# classification report

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=[
            "AI",
            "Agents",
            "DB",
            "HCI",
            "IR",
            "ML"
        ]
    )
)

              precision    recall  f1-score   support

          AI       0.53      0.42      0.47        50
      Agents       0.79      0.87      0.83       119
          DB       0.84      0.79      0.82       140
         HCI       0.79      0.80      0.80       102
          IR       0.74      0.69      0.72       134
          ML       0.68      0.75      0.71       118

    accuracy                           0.75       663
   macro avg       0.73      0.72      0.72       663
weighted avg       0.75      0.75      0.75       663



In [20]:
# permutation-invariance test

test_sample_index = None

for index in test_indices:
    node_id = node_ids[index]

    if len(neighbors[node_id]) > 1:
        test_sample_index = index
        break

print("Selected test index:", test_sample_index)
print(
    "Number of neighbors:",
    len(neighbors[node_ids[test_sample_index]])
)

Selected test index: 1614
Number of neighbors: 2


In [21]:
# compare original and shuffled neighborhood
model.eval()

center, neighbor_set, label = get_node_data(
    node_ids[test_sample_index]
)

center_tensor = torch.tensor(
    center,
    dtype=torch.float32
)

neighbor_tensor = torch.tensor(
    neighbor_set,
    dtype=torch.float32
)

# Original ordering
with torch.no_grad():
    original_output = model(
        center_tensor,
        neighbor_tensor
    )

    original_probabilities = torch.softmax(
        original_output,
        dim=0
    )

# Shuffled ordering
permutation = torch.randperm(
    neighbor_tensor.size(0)
)

shuffled_tensor = neighbor_tensor[
    permutation
]

with torch.no_grad():
    shuffled_output = model(
        center_tensor,
        shuffled_tensor
    )

    shuffled_probabilities = torch.softmax(
        shuffled_output,
        dim=0
    )

difference = torch.abs(
    original_probabilities -
    shuffled_probabilities
)

print("Original probabilities:")
print(original_probabilities)

print("\nShuffled probabilities:")
print(shuffled_probabilities)

print(
    "\nMaximum difference:",
    difference.max().item()
)

Original probabilities:
tensor([2.7451e-08, 9.1941e-12, 1.0000e+00, 5.6537e-16, 8.7851e-12, 1.0910e-09])

Shuffled probabilities:
tensor([2.7451e-08, 9.1941e-12, 1.0000e+00, 5.6537e-16, 8.7851e-12, 1.0910e-09])

Maximum difference: 0.0


In [22]:
# test 10 random permutations
permutation_differences = []

with torch.no_grad():

    original_output = model(
        center_tensor,
        neighbor_tensor
    )

    original_probabilities = torch.softmax(
        original_output,
        dim=0
    )

    for _ in range(10):

        permutation = torch.randperm(
            neighbor_tensor.size(0)
        )

        shuffled_tensor = neighbor_tensor[
            permutation
        ]

        shuffled_output = model(
            center_tensor,
            shuffled_tensor
        )

        shuffled_probabilities = torch.softmax(
            shuffled_output,
            dim=0
        )

        difference = torch.abs(
            original_probabilities -
            shuffled_probabilities
        )

        permutation_differences.append(
            difference.max().item()
        )

print("Permutation differences:")
print(permutation_differences)

print(
    "Maximum difference:",
    max(permutation_differences)
)

Permutation differences:
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Maximum difference: 0.0
